# Quick DAXM grid: grains along the beam, strain and lattice parameters averaged over their thickness

All the wire scans of a raster / grid (one DAXM wire scan per point), without reconstruction of the images:

| step | what | result |
|---|---|---|
| 1 | wire scans of the grid | |
| 2 | wires calibration (notebook `1_wires_calibration`) and its **p0 shift for each scan**: the wires parameters are those of the calibration; p0 of all the wires is shifted by the change of the position of the sample surface along the beam, measured on the dips of each scan (envelope of the shallowest grains) | `<scan>/daxm/wirescalib/wires_grid.calib` |
| 3 | **quick daxm of every scan** (in this notebook, or jobs on SLURM nodes: `magnifix3` = 3 jobs of one node) | `<scan>/daxm/quickdaxm.h5`, `grid_summary.json` |
| 4 | maps of the checks: surface offset, p0 shift, spots with a depth, **2-wire spread** (consistency of the calibration at each scan), mismatch | |
| 5 | **depth layers** of each scan: peaks of the distribution of the depths of its spots (the grains along the beam); one peak list per layer, 2theta and chi of each spot from its own depth | `<grid folder>/layers/layer_<scan position><layer>.cor` |
| 6 | **indexation and strain refinement of the layers** (LaueTools indexation, one or several materials): orientation, deviatoric strain, lattice parameters averaged over the thickness of the grain(s) of each layer | `<grid folder>/fitfiles_<material>` |
| 7 | browser of the peak lists of the layers (gnomonic projection) | |

Results of the grid: `PROCESSED_DATA/<sample>/<GRID_NAME>`. Running the notebook again is safe: scans already done are reloaded (`REDO = True` to compute again).

Accurate depths (and strain): the calibration of notebook 1 refined with the fluorescence, and the p0 shift of each scan. The 2-wire spread of each scan tells where the calibration holds (a few microns).

# SETTINGS

In [ ]:
# 1. experiment and grid
RAW_DATA_FOLDER = '/data/visitor/<proposal>/bm32/<date>/RAW_DATA'
GRID_FILTER = 'Zr5_griddaxm_grid -0p05sec -1sec -2sec'  # words of the dataset names of the wire scans ('-word': without it)
GRID_NAME = None           # folder of the grid results in PROCESSED_DATA/<sample> (None: <sample>_quickdaxm_grid)
MAX_SCANS = None           # test on the first scans only (e.g. 28: first row of the grid); None: all
MODE = 'compute'           # 'compute'; 'show': only show existing results

# 2. calibrations
DETFILE = 'auto'           # detector calibration .det file; 'auto': the last one acquired before the first scan
CALIBFILE = 'auto'         # wires calibration: 'auto': made with a wire scan of the same sample folder (notebook
                           # 1_wires_calibration), or a Ge wire scan of the folder; or path of a .calib file
P0_OFFSET = 'envelope'     # p0 of the wires shifted for each scan ('envelope'), or None: calibration as it is
OFFSET_OUTLIERS = 0.02     # fraction of the dips of a wire allowed above its shallowest sources

# 3. quick daxm and depth layers
QUICKDAXM = {'skip_images': 40}  # changes of the quick daxm parameters. skip_images: first images of the scans not used
                           # (Zr5 grid: wire position wrong by ~8 µm in images 0-39 of the scans after the first one,
                           # see mismatch_vs_image())
LAYER_BANDWIDTH = 2.       # microns, smoothing of the distribution of the depths of the spots of a scan
LAYER_MIN_SPOTS = 8        # layers with fewer spots left out
LAYER_PROMINENCE = 0.1     # min prominence of a layer (fraction of the highest peak of the distribution)

# 4. indexation of the layers
MATERIALS = ['Zr', 'Cr']   # materials indexed (each layer indexed with each material)
NB_GRAINS = 2              # max nb of grains per layer
INDEXING = {}              # other indexation parameters, e.g. {'e_max': 22} (configs/indexing_<material>.yaml of the grid)

# 5. computation
RESOURCE = 'kernel'        # quick daxm of the scans: 'kernel' (this notebook), 'local', 'magnifix', 'magnifix3' (3 jobs) ...
NB_CPUS = None             # None: all available cpus
NB_JOBS = None             # SLURM: nb of jobs (default: nb of nodes of RESOURCE)
TIME_LIMIT = '02:00:00'    # SLURM time limit of a job
REDO = False               # True: compute again the quick daxm of the scans
OUTPUT_ROOT = None         # None: PROCESSED_DATA of the experiment

# IMPORTS

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
try:
    import ipympl  # interactive plots (zoom, hover) in JupyterLab / VS Code
    %matplotlib widget
except ImportError:
    %matplotlib inline
from IPython.display import display

import LaueTools.daxm_workflow as DW
import LaueTools.Daxm.quickdaxm as QD
import LaueTools.generaltools as GT

# STEP 1: wire scans of the grid

In [ ]:
wirescans = DW.list_wire_scans(RAW_DATA_FOLDER, contains=GRID_FILTER, output_root=OUTPUT_ROOT)
grid = wirescans[wirescans['end'] == 'SUCCESS']
rowcol = np.array([DW.grid_position(d) for d in grid['dataset']], dtype=float)
if np.all(np.isfinite(rowcol)):   # grid order: rows, then columns
    grid = grid.iloc[np.lexsort((rowcol[:, 1], rowcol[:, 0]))]
if MAX_SCANS:
    grid = grid.iloc[:MAX_SCANS]
datasets = list(grid['dataset'])
first = int(grid.index[0])
print(f"{len(datasets)} wire scans of the grid (of {len(wirescans)} found), first: {datasets[0]}, last: {datasets[-1]}")
if np.all(np.isfinite(rowcol)):
    print(f'grid of {int(rowcol[:, 0].max()) + 1} rows x {int(rowcol[:, 1].max()) + 1} columns')
grid[['dataset', 'start', 'command', 'end', 'done']].head()

# STEP 2: calibrations

Wires calibration (`CALIBFILE`) and reference of the p0 shifts: the scan the calibration was made with (its own envelope offset), else the first scan of the grid.

In [ ]:
detfile = DW.detfile_for_scan(wirescans, first) if str(DETFILE) == 'auto' else DETFILE
calibfile = DW.resolve_calibfile(wirescans, first, CALIBFILE, detfile=detfile, nb_cpus=NB_CPUS)
if calibfile is None:
    GT.printred('no wires calibration: run notebook 1_wires_calibration first (or set CALIBFILE)')
folder = DW.grid_folder(wirescans, first, GRID_NAME) if OUTPUT_ROOT is None else Path(OUTPUT_ROOT) / (GRID_NAME or 'quickdaxm_grid')
settings = dict(raw_data_folder=RAW_DATA_FOLDER, contains=GRID_FILTER, output_root=OUTPUT_ROOT, grid_folder=str(folder),
                datasets=datasets, detfile=str(detfile), calibfile=str(calibfile), p0_offset=P0_OFFSET,
                offset_outliers=OFFSET_OUTLIERS, quickdaxm=QUICKDAXM, redo=REDO, layer_bandwidth=LAYER_BANDWIDTH,
                layer_min_spots=LAYER_MIN_SPOTS, layer_prominence=LAYER_PROMINENCE)
if P0_OFFSET == 'envelope' and calibfile is not None and MODE != 'show':
    settings['reference_offset'] = DW.reference_offset(wirescans, calibfile, first, NB_CPUS, settings)
print('grid results folder:', folder)

# STEP 3: quick daxm of all the wire scans

Each scan: dips of its Laue spots, p0 shift, depths, checks, depth layers. `RESOURCE = 'kernel'`: one scan after the other in this notebook (about 5-20 s per scan); `'magnifix3'`: the scans shared between 3 jobs of one node each (the notebook waits for them).

In [ ]:
if MODE == 'show':
    summary = DW.load_grid_summary(settings)
else:
    summary = DW.run_quickdaxm_grid(settings, resource=RESOURCE, nb_cpus=NB_CPUS, nb_jobs=NB_JOBS, time_limit=TIME_LIMIT)
print(f'{len(summary)} scans done')
if 'error' in summary:
    errors = summary[summary['error'].notna()]
    if len(errors):
        GT.printred(f'{len(errors)} scans failed:'); display(errors[['dataset', 'error']])
summary[['dataset', 'row', 'col', 'offset', 'p0_shift', 'spots_with_depth', 'spread', 'mismatch', 'ybeam_median', 'nb_layers']].round(2)

# STEP 4: maps of the checks

- `offset`: position along the beam (µm) of the shallowest grains of each scan with the calibration: the surface of the sample along the beam (its variations = `p0_shift`, applied to the wires of each scan)
- `spots_with_depth`, `nb_layers`, `ybeam_median`: the depth analysis of each scan
- `spread`: median difference of the depths given by two wires for the same spot (consistency of the calibration at each scan, a few microns expected); `mismatch`: median difference of the depths of the entry and exit edges of the dips

In [ ]:
MAPS = ['offset', 'p0_shift', 'spots_with_depth', 'spread', 'mismatch', 'nb_layers', 'ybeam_median']

plt.close('all')
if len(summary) and summary['row'].notna().all():
    ncols = 4
    fig, axes = plt.subplots(int(np.ceil(len(MAPS) / ncols)), ncols, figsize=(4.5 * ncols, 3.2 * int(np.ceil(len(MAPS) / ncols))))
    for ax in axes.ravel()[len(MAPS):]:
        ax.set_visible(False)
    for ax, column in zip(axes.ravel(), MAPS):
        values = DW.grid_map(summary, column)
        vmin, vmax = np.nanpercentile(values, [2, 98]) if np.isfinite(values).any() else (None, None)
        im = ax.imshow(values, origin='lower', vmin=vmin, vmax=vmax, cmap='viridis')
        ax.set_title(column, fontsize=9)
        ax.set_xlabel('column'); ax.set_ylabel('row')
        fig.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout()
else:
    print('no row/column in the dataset names: maps not available (see the table above)')
plt.show()

# STEP 5: depth layers of the scans

One line per layer: scan (row, col), layer index, range and median of the depths of its spots (`ybeam`, µm along the beam), `extent` (std), nb of spots, peak list. Left: depths of the layers along the grid (scan position); right: their distribution.

In [ ]:
layers = DW.load_grid_layers(settings)
print(f'{len(layers)} layers in {layers["dataset"].nunique() if len(layers) else 0} scans')

plt.close('all')
if len(layers):
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
    sc = axes[0].scatter(layers['position'], layers['ybeam'], s=6 + 0.05 * layers['nb_spots'], c=layers['extent'], cmap='viridis')
    axes[0].set_xlabel('scan position in the grid'); axes[0].set_ylabel('ybeam of the layer (µm)')
    fig.colorbar(sc, ax=axes[0], label='extent (µm)')
    axes[1].hist(layers['ybeam'], bins=np.arange(np.floor(layers['ybeam'].min()) - 2, np.ceil(layers['ybeam'].max()) + 2, 2.), weights=layers['nb_spots'])
    axes[1].set_xlabel('ybeam of the layer (µm)'); axes[1].set_ylabel('nb of spots')
    fig.tight_layout()
plt.show()
layers.head(10)

# STEP 6: indexation of the layers

Each layer (peak list of the spots of the grain(s) at one depth) indexed and refined with each material of `MATERIALS` (`NB_GRAINS` grains at most): `configs/indexing_<material>.yaml` of the grid (written at the first run, then kept: edit it for other parameters), `.fit` files in `fitfiles_<material>`. Table: nb of indexed spots, mean pixel deviation, deviatoric strain (10⁻³, sample frame), new lattice parameters.

In [ ]:
INDEX_LAYERS = True

results_layers = {}
if INDEX_LAYERS and len(layers):
    for material in MATERIALS:
        if MODE != 'show':
            DW.index_layers(settings, material, nb_grains=NB_GRAINS, nb_cpus=NB_CPUS, detfile=detfile, **INDEXING)
        results_layers[material] = DW.layers_results(settings, material)
        table = results_layers[material]
        print(f"{material}: {int(table['indexed'].notna().sum())} / {len(table)} layers indexed")
for material, table in results_layers.items():
    cols = [c for c in ('row', 'col', 'layer', 'ybeam', 'nb_spots', 'indexed', 'pixdev', 'eps_sample_xx', 'eps_sample_yy',
                        'eps_sample_zz', 'a', 'b', 'c') if c in table]
    print(material); display(table[table['indexed'].notna()][cols].round(4).head(20))

In [ ]:
# deviatoric strain and lattice parameters of the indexed layers vs their depth (one color per material)
QUANTITIES = ['eps_sample_xx', 'eps_sample_yy', 'eps_sample_zz', 'a', 'c']
MAX_PIXDEV = 1.0   # layers with a larger mean pixel deviation left out

plt.close('all')
if results_layers:
    quantities = [q for q in QUANTITIES if any(q in t for t in results_layers.values())]
    fig, axes = plt.subplots(1, len(quantities), figsize=(4 * len(quantities), 3.5))
    for ax, quantity in zip(np.atleast_1d(axes), quantities):
        for material, table in results_layers.items():
            good = table[(table['pixdev'] <= MAX_PIXDEV)] if 'pixdev' in table else table
            if quantity in good:
                ax.plot(good['ybeam'], good[quantity], 'o', ms=3, label=material)
        ax.set_xlabel('ybeam (µm)'); ax.set_title(quantity, fontsize=9)
    np.atleast_1d(axes)[0].legend(fontsize=8)
    fig.tight_layout()
plt.show()

In [ ]:
# map of a quantity of the layers in a depth range (mean over the layers of each scan), e.g. strain of the Zr grains
MAP_MATERIAL = MATERIALS[0]
MAP_QUANTITY = 'eps_sample_zz'
MAP_YBEAM_RANGE = (0, 100)   # µm

plt.close('all')
table = results_layers.get(MAP_MATERIAL)
if table is not None and len(table) and MAP_QUANTITY in table and table['row'].notna().all():
    sel = table[(table['ybeam'] >= MAP_YBEAM_RANGE[0]) & (table['ybeam'] < MAP_YBEAM_RANGE[1]) & (table['pixdev'] <= MAX_PIXDEV)]
    mean = sel.groupby(['row', 'col'], as_index=False)[MAP_QUANTITY].mean()
    if len(mean):
        fig, ax = plt.subplots(figsize=(8, 4))
        values = DW.grid_map(mean, MAP_QUANTITY)
        im = ax.imshow(values, origin='lower', cmap='RdBu_r', vmin=-np.nanmax(np.abs(values)), vmax=np.nanmax(np.abs(values)))
        ax.set_title(f'{MAP_MATERIAL} {MAP_QUANTITY}, layers {MAP_YBEAM_RANGE} µm', fontsize=9)
        ax.set_xlabel('column'); ax.set_ylabel('row')
        fig.colorbar(im, ax=ax)
plt.show()

# STEP 7: browser of the peak lists of the layers

Peak lists of the layers of one scan (slider), on the detector and in gnomonic projection: the spots of a grain lie on lines (zone axes). Grey: all the spots of the scan.

In [ ]:
SCAN_SHOWN = datasets[0]   # dataset name of the scan

plt.close('all')
if len(layers):
    files = list(layers[layers['dataset'] == SCAN_SHOWN]['file'])
    allspots = Path(DW.workdir(wirescans.loc[DW.scan_index(wirescans, SCAN_SHOWN), 'image_folder'], OUTPUT_ROOT)) / 'quickdaxm_spots.cor'
    if files:
        browser = QD.PeakListBrowser(files, nbmax=200, background_file=str(allspots) if allspots.exists() else files[0])